# 02 — Cobertura DIEST (corpus-alvo da Fase 2)

Filtra `data/interim/metadados.parquet` para o subset da **Diretoria de
Estudos e Políticas do Estado, das Instituições e da Democracia (DIEST)**
e descreve o corpus-alvo que irá para a Fase 2 (download de PDFs + Docling).

O filtro combina dois sinais:
- `orgunit_uuid == 38b33462-dc75-43f2-9bda-1f3c54757ae7` (linkagem nova,
  típica de itens pós-2020);
- `contributor_other` casando com `DIEST` ou `Diretoria de Estudos e
  Políticas do Estado` (histórico completo).

O subset é persistido em `data/interim/metadados_diest.parquet`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))
from src.scraping import filter_by_diretoria

SRC = Path('../data/interim/metadados.parquet')
DEST = Path('../data/interim/metadados_diest.parquet')
df = pd.read_parquet(SRC)
print(f'Corpus completo: {len(df):,} documentos')

In [ ]:
diest = filter_by_diretoria(df, 'DIEST')
diest.to_parquet(DEST, index=False)
print(f'DIEST: {len(diest):,} documentos ({len(diest)/len(df):.1%} do corpus)')
print(f'Escrito em {DEST}')

## Quebra dos sinais usados no filtro

Quantos itens foram capturados por UUID vs. por texto livre.

In [ ]:
DIEST_UUID = '38b33462-dc75-43f2-9bda-1f3c54757ae7'
by_uuid = df['orgunit_uuid'].fillna('').str.contains(DIEST_UUID, regex=False)
by_text = df['contributor_other'].fillna('').str.contains(
    r'(?:\bDIEST\b|Diretoria de Estudos e Políticas do Estado)',
    case=False, regex=True,
)
print(f'Por orgunit_uuid:       {int(by_uuid.sum()):>6}')
print(f'Por contributor_other:  {int(by_text.sum()):>6}')
print(f'Por ambos:              {int((by_uuid & by_text).sum()):>6}')
print(f'União (DIEST total):    {int((by_uuid | by_text).sum()):>6}')

## Cobertura temporal e de PDF

In [ ]:
anos = diest['ano'].dropna().astype(int)
print(f'Ano mínimo: {anos.min()}')
print(f'Ano máximo: {anos.max()}')
print(f'Sem ano:    {diest["ano"].isna().sum()}')
print()
print(f'Com handle:  {diest["handle"].notna().sum():>6} ({diest["handle"].notna().mean():.1%})')
print(f'Com resumo:  {(diest["resumo"] != "").sum():>6} ({(diest["resumo"] != "").mean():.1%})')

In [ ]:
print('=== Tipos documentais (DIEST) ===')
print(diest['tipo'].fillna('(sem tipo)').value_counts().to_string())

In [ ]:
d = diest.dropna(subset=['ano']).copy()
d['decada'] = (d['ano'].astype(int) // 10) * 10
print('=== Documentos DIEST por década ===')
print(d['decada'].value_counts().sort_index().to_string())
ax = d['ano'].astype(int).value_counts().sort_index().plot(
    kind='bar', figsize=(14, 4), title='DIEST — documentos por ano'
)
ax.set_xlabel('Ano'); ax.set_ylabel('N');

In [ ]:
autores = diest['autores'].fillna('').str.split('; ').explode().str.strip()
autores = autores[autores != '']
print('=== Top 30 autores DIEST ===')
print(autores.value_counts().head(30).to_string())

In [ ]:
palavras = diest['palavras_chave'].fillna('').str.split(', ').explode().str.strip()
palavras = palavras[palavras != '']
print('=== Top 40 palavras-chave DIEST ===')
print(palavras.value_counts().head(40).to_string())

In [ ]:
pivot = d.groupby(['decada', 'tipo']).size().unstack(fill_value=0)
pivot = pivot.loc[:, pivot.sum().sort_values(ascending=False).head(8).index]
print('=== Tipo × década (DIEST) ===')
pivot